# Exploring the *A. oryzae* secretion benchmarkA short tour of the dataset in `data/`, aimed at someone deciding what toengineer next.The data is split across four CSVs because that is the shape the biologyactually has: a study reports several experiments, an experiment may editseveral genes at once, and each experiment produces several measurements.That normalisation is what keeps the records honest, but it also means thequestions people actually ask do not map onto any single file. Every querybelow is a join.This notebook only reads from `data/`. It never writes to it.

In [1]:
import pandas as pd

DATA = "../data"

# Read everything as text: the tables use "TODO" and "not_reported" as real
# values, and letting pandas coerce them into NaN would hide the difference
# between "nobody has checked yet" and "the paper does not say".
def load(name):
    return pd.read_csv(f"{DATA}/{name}", dtype=str, keep_default_na=False)

studies = load("studies.csv")
experiments = load("experiments.csv")
genes = load("experiment_genes.csv")
outcomes = load("outcomes.csv")

for name, frame in [("studies", studies), ("experiments", experiments),
                    ("experiment_genes", genes), ("outcomes", outcomes)]:
    print(f"{name:<18} {len(frame):>3} rows x {frame.shape[1]} cols")

studies             15 rows x 8 cols
experiments         11 rows x 8 cols
experiment_genes    13 rows x 6 cols
outcomes            23 rows x 12 cols


## Query 1 — What has been tried, by strategy**The question.** Of all the interventions in the literature so far, what*kinds* of thing have people actually tried? Which strategies are wellexplored, and which has the field barely touched?**Why the raw files do not answer it.** `gene_role` — the strategy label —lives in `experiment_genes.csv`, but the thing worth counting is*experiments*, and that lives in `experiments.csv`. An experiment that editstwo genes has two rows in the gene table, so counting rows would quietlyweight double disruptions twice. The count has to be over distinct`(experiment_id, gene_role)` pairs, not over rows.

In [2]:
by_strategy = (
    genes.merge(experiments, on="experiment_id")
         .drop_duplicates(["experiment_id", "gene_role"])
         .groupby("gene_role")
         .agg(experiments=("experiment_id", "nunique"),
              studies=("study_id", "nunique"))
         .sort_values("experiments", ascending=False)
)

print(by_strategy)

                  experiments  studies
gene_role                             
remove_protease             8        2
fix_misrouting              2        1
target_regulator            2        1


A strategy that edits more than one gene in a single experiment is countedonce under each role it uses, which is why the column can sum to more thanthe number of experiments.The shape of this table is the point. The dataset is small, and it islopsided: nearly everything tried so far has been deleting a protease. TheREADME lists eight strategy categories; most of them have no rows at all.That absence is a finding — it says where the literature has *not* looked,which is at least as useful for planning as knowing what worked.

## Query 2 — Has anyone tried this gene?**The question.** This is the reference use case for the whole dataset.Before committing to a knockout, you want to know whether someone alreadydid it, what they expressed, what they compared against, and what they got.**Why the raw files do not answer it.** The answer is spread across threetables. The gene name is in `experiment_genes.csv`; the cargo protein andcontrol strain are in `experiments.csv`; the numbers are in`outcomes.csv`. There is no single row anywhere that holds all of it.

In [3]:
def has_anyone_tried(gene_name):
    """Every experiment that edited `gene_name`, with its setup and results."""
    hits = genes[genes["gene_name"].str.lower() == gene_name.lower()]
    if hits.empty:
        print(f"No experiment in the dataset edits {gene_name!r}.")
        return None

    setup = hits.merge(experiments, on="experiment_id").merge(
        studies[["study_id", "authors", "year"]], on="study_id")

    return (
        setup.merge(outcomes, on="experiment_id")
             .assign(first_author=lambda d: d["authors"].str.split(";").str[0])
             [["experiment_id", "first_author", "year", "cargo", "edit_notation",
               "control_strain", "strain", "arm", "value", "unit", "vs_control",
               "source_ref"]]
             .sort_values(["experiment_id", "arm"])
             .reset_index(drop=True)
    )


tppa = has_anyone_tried("tppA")
print(tppa.to_string())

       experiment_id first_author  year           cargo edit_notation control_strain       strain       arm value  unit vs_control source_ref
0       JIN2007_TPPA       Jin FJ  2007  human lysozyme   ΔtppA::adeA          NA-2L        NA-2L   control  15.6  mg/L       1.0x     Fig. 4
1       JIN2007_TPPA       Jin FJ  2007  human lysozyme   ΔtppA::adeA          NA-2L    NA-2L-tp5  modified  21.2  mg/L      1.36x     Fig. 4
2  JIN2007_TPPA_PALB       Jin FJ  2007  human lysozyme   ΔtppA::argB           N-2L   N2L-pa-tp6  modified  TODO  mg/L       TODO     Fig. 5
3  JIN2007_TPPA_PEPE       Jin FJ  2007  human lysozyme   ΔtppA::argB           N-2L         N-2L   control  TODO  mg/L       1.0x     Fig. 5
4  JIN2007_TPPA_PEPE       Jin FJ  2007  human lysozyme   ΔtppA::argB           N-2L  N2L-peE-tp6  modified  25.4  mg/L      1.63x     Fig. 5


Two experiments touch `tppA`: one on its own, one paired with `pepE`. Notethat they are measured against *different* control strains, so theirfold-changes are not two readings of the same quantity. Note too that somevalues are still `TODO` — the row exists and the strain is named, but thenumber has not yet been read off the figure. A query like this one has toshow that state rather than drop it, or the caller will mistake "not yettranscribed" for "not measured".One trap worth knowing about before you write your own version of thisquery: **not every experiment has a `control` row in `outcomes.csv`.** When apaper measures one control once and reads several disruptants against it,that control is stored once, under a single experiment, rather than copiedacross all of them — duplicating it would record one measurement as five.The link that always holds is `experiments.control_strain`, which names thecontrol for every experiment whether or not a row for it sits under that`experiment_id`. Join on the strain name, not on the presence of a controlrow.

## Query 3 — Multi-gene experiments**The question.** Which experiments changed more than one gene at once, andwhat did those combinations produce?**Why this one matters most.** This query is the reason the schema is shapedthe way it is. A double disruption is **one experiment with two gene rowsand one measurement** — not two experiments.You cannot recover the individual contribution of each gene from a strainthat is missing both. The strain has one phenotype, and it is the jointresult. A flatter schema — one row per experiment with a `gene` column —would force you to either invent two rows that were never separatelymeasured, or throw one gene away. Both are lies about what the paper did.Splitting genes into their own table lets one experiment carry many genechanges and still point at a single number.`JIN2007_TPPA_PALB` is the case that proves the point: it produced *less*lysozyme than either single disruption did. Two edits that each helpindividually hurt in combination. Nothing about the single-gene rowspredicts that, which is exactly why the combination has to be its ownrecord.

In [4]:
gene_counts = genes.groupby("experiment_id").size()
multi_ids = gene_counts[gene_counts > 1].index

print(f"{len(multi_ids)} of {len(gene_counts)} experiments edit more than one gene\n")

for experiment_id in multi_ids:
    edits = genes[genes["experiment_id"] == experiment_id]
    setup = experiments[experiments["experiment_id"] == experiment_id].iloc[0]

    edit_list = ", ".join(f"{r.gene_name} ({r.gene_role})" for r in edits.itertuples())
    print(f"{experiment_id}  -  {len(edits)} genes: {edit_list}")
    print(f"  cargo: {setup.cargo}   control: {setup.control_strain}")

    results = outcomes[outcomes["experiment_id"] == experiment_id]
    print(results[["strain", "arm", "value", "unit", "vs_control", "source_ref"]]
          .to_string(index=False))
    print()

2 of 11 experiments edit more than one gene

JIN2007_TPPA_PALB  -  2 genes: tppA (remove_protease), palB (target_regulator)
  cargo: human lysozyme   control: N-2L
    strain      arm value unit vs_control source_ref
N2L-pa-tp6 modified  TODO mg/L       TODO     Fig. 5

JIN2007_TPPA_PEPE  -  2 genes: tppA (remove_protease), pepE (remove_protease)
  cargo: human lysozyme   control: N-2L
     strain      arm value unit vs_control source_ref
       N-2L  control  TODO mg/L       1.0x     Fig. 5
N2L-peE-tp6 modified  25.4 mg/L      1.63x     Fig. 5



## Query 4 — Fold-change by strategy**The question.** Grouped by strategy, what size of effect has each kind ofintervention produced?**Why the raw files do not answer it.** `vs_control` is stored as text(`2.9x`), because that is how it reads in the source, so it has to be parsedbefore it can be aggregated. The strategy label is in a different fileagain. And rows still awaiting transcription hold `TODO`, which has tobecome a missing value rather than a zero.**A warning about this table.** These numbers are **not comparable acrossstudies.** Each study used its own construct, its own control strain, andits own culture conditions — `JIN2007` alone expresses two tandem copies ofthe cargo where `ZHU2012` and `YOON2010` express one, so its milligramfigures are on a different scale entirely. A fold-change here says "thisedit beat *its own* control by this much", nothing more.The column exists to spot patterns worth investigating. It is not aleaderboard, and sorting by it does not rank interventions.

In [5]:
def as_number(value):
    """'2.9x' -> 2.9; 'TODO' and blanks -> missing."""
    try:
        return float(str(value).rstrip("x"))
    except ValueError:
        return float("nan")


modified = outcomes[outcomes["arm"] == "modified"].copy()
modified["fold"] = modified["vs_control"].map(as_number)

# A multi-gene experiment joins to one row per gene, so it appears under each
# strategy it used. Deduplicate per (outcome, role) so a single measurement is
# not counted twice within one strategy.
scored = (modified.merge(genes[["experiment_id", "gene_role"]], on="experiment_id")
                  .drop_duplicates(["outcome_id", "gene_role"]))

summary = (scored.groupby("gene_role")
                 .agg(measurements=("outcome_id", "count"),
                      with_a_number=("fold", "count"),
                      median_fold=("fold", "median"),
                      best_fold=("fold", "max"))
                 .sort_values("median_fold", ascending=False))

print(summary.round(2))
print()
print(scored[["outcome_id", "experiment_id", "gene_role", "strain", "value",
              "unit", "vs_control"]].sort_values("gene_role").to_string(index=False))

                  measurements  with_a_number  median_fold  best_fold
gene_role                                                            
fix_misrouting               4              4         2.35       3.00
remove_protease             12              8         1.66       2.90
target_regulator             2              1         1.14       1.14

  outcome_id     experiment_id        gene_role        strain value unit vs_control
YOON2010_005      YOON2010_HLY   fix_misrouting   SlDv10-HLY1  22.6 mg/L       2.0x
YOON2010_002      YOON2010_CHY   fix_misrouting   SlDv10-AKC3  83.1 mg/L       3.0x
YOON2010_003      YOON2010_CHY   fix_misrouting   SlDv10-AKC4  70.3 mg/L       2.5x
YOON2010_006      YOON2010_HLY   fix_misrouting   SlDv10-HLY2  24.6 mg/L       2.2x
 JIN2007_008 JIN2007_TPPA_PEPE  remove_protease   N2L-peE-tp6  25.4 mg/L      1.63x
 JIN2007_005      JIN2007_ALPA  remove_protease     NA-2L-al3  TODO mg/L       TODO
 JIN2007_004      JIN2007_PEPE  remove_protease   NA-2L-peE10

## What this dataset cannot answer yetThe queries above run, but the honest summary is that the dataset is far toosmall to support most of what someone would want to ask of it.- **It cannot rank strategies.** With only a handful of studies, and  nearly all of them deleting proteases, any apparent ordering between  categories is a statement about which papers happen to be curated, not  about which strategy works better.- **It cannot compare across studies.** Different constructs, cargo copy  numbers, control strains, and culture conditions mean the fold-change  column is only meaningful within a single experiment's own comparison.  There is no shared baseline to normalise against.- **It cannot predict combinations.** `JIN2007_TPPA_PALB` is the whole  argument: two edits that each help individually produced *less* than  either alone. One counterexample is enough to rule out additivity as a  safe assumption, but nowhere near enough to model what actually happens.- **It cannot separate cargo from host.** Chymosin and lysozyme respond  differently to the same edit, and the dataset has too few cargo proteins  to tell a cargo-specific effect from a general one.- **It is not yet fully transcribed.** Several values are still `TODO`,  read off bar charts that have not been digitised. Any aggregate over  `vs_control` is computed on the subset that has numbers, which is not a  random subset — the values stated in paper text got transcribed first.The useful thing the dataset already does is the negative space: it makesvisible which strategies the literature has barely tried, and it recordscombinations and failed interventions that a summary of headline resultswould drop.